# The lazy `.gpkg` form, against a real model

`mf.chd.gpkg(path, layer=..., nper=...)` can now be declared **without** `context=` and
resolves its cells against whatever grid the build supplies (ledger 279), freezing what it
built so a run stays self-contained (ledger 280). One declaration works on any mesh, which
is what turns a `move_report` row from `stale` into `rebuilt`.

It passes the full suite — on 36- and 100-cell grids I made up, with features drawn to be
convenient. Ten Trails is 20,323 Voronoi cells in State Plane, with boundaries digitised
by somebody solving an actual problem. Only one of those is evidence.

This notebook works through the whole thing: importing a finished run back into a project,
what the model declares, the layer geometry a `CellSurfaceOffset` resolves against, all
seven boundary helpers, the freeze, and the cost.

**It never writes to the model directory.** Everything is copied into a working directory
first, and the last cell checks.

In [1]:
import json, pathlib, shutil, tempfile, time
import numpy as np
import pandas as pd
import geopandas as gpd
import myflopy as mf
from myflopy.specs import ModelContext, package_move

MODEL_ROOT = pathlib.Path("/home/lukem/models/mf6/tentrails")
DATA_ROOT = pathlib.Path("/home/lukem/models/mf6/Ten Trails/data")
SOURCE_RUN = MODEL_ROOT / "runs" / "sw_mounding_flow"

assert MODEL_ROOT.exists(), f"this notebook needs the private Ten Trails model at {MODEL_ROOT}"

WORK = pathlib.Path(tempfile.mkdtemp(prefix="lazy_gpkg_"))
INPUTS = WORK / "inputs"; INPUTS.mkdir(parents=True)
print("working directory:", WORK)

working directory: /tmp/lazy_gpkg_4enxynub


---
# Part 1 — Importing a finished run

Before any of the lazy work: a run on disk is a complete record of a model, and it is
worth knowing exactly how much of one you can get back.

There is no `import_run` verb — that is planned, not built. What exists is `load_run`,
which reopens a run from its workspace, plus the run's own copy of every library package
it used (`<run>/_library/`, ledger 230). Between them you can reconstruct the project.

Copy a real run first, so nothing below can touch the original.

In [2]:
RUN = WORK / "sw_mounding_flow"
t = time.perf_counter()
shutil.copytree(SOURCE_RUN, RUN)
print(f"copied {sum(f.stat().st_size for f in RUN.rglob('*') if f.is_file()) / 1e6:.0f} MB "
      f"in {time.perf_counter() - t:.1f} s")

run = mf.load_run(RUN)
print(f"\nrun      : {run.name}")
print(f"status   : {run.status}          <- myflopy's verb: was it written, executed?")
print(f"outcome  : {run.outcome}   <- MF6's verdict, read from mfsim.lst")

copied 414 MB in 0.1 s



run      : sw_mounding_flow
status   : written          <- myflopy's verb: was it written, executed?
outcome  : ListingOutcome(outcome='normal', elapsed='1 Hours,  0 Minutes, 16 Seconds')   <- MF6's verdict, read from mfsim.lst


`status` and `outcome` answer different questions, which is why there are two of them:
`status` is what myflopy did, `outcome` is what MODFLOW said about it (ledger 179-191).

Now the spec. The run recorded the simulation it was built from, with its packages as
library **keys** rather than inlined copies.

In [3]:
spec = run.spec
print(f"{type(spec).__name__}: {spec.name}")
for model in spec.models:
    print(f"  model {model.name!r} ({model.type_enum.value})")
    for package in model.packages:
        print(f"      {getattr(package, 'key', getattr(package, 'name', '?'))}")
print(f"\nsimulation packages: {[getattr(p, 'key', getattr(p, 'name', '?')) for p in spec.packages]}")
print(f"recorded digests   : {len(run.package_digests or {})}")

SimulationSpec: sw_mounding_flow
  model 'tentrails' (gwf)
      dis/base
      npf/base
      sto/base
      ic/steady
      chd/base
      ghb/base
      drn/base
      riv/rock
      crisp/base
      oc/all
      lak/sw
      rch/background

simulation packages: ['tdis', 'ims']
recorded digests   : 12


Those keys point at packages that lived in a project which may since have moved on — or
may not exist any more. That is what the run's private library is for: a prepared run
copies every package it used, so it can be rebuilt as it ran rather than as the library
now stands.

In [4]:
from myflopy.workspace import RUN_LIBRARY_DIR, _revive_package

library_dir = RUN / RUN_LIBRARY_DIR / "packages"
library = {}
for path in sorted(library_dir.glob("*/*.json")):
    if "versions" in path.name:
        continue
    key = f"{path.parent.name}/{path.stem}"
    library[key] = _revive_package(
        key, json.loads(path.read_text()), root=RUN,
        sidecar_path=path.with_name(f"{path.stem}.versions.json"), stacklevel=3,
    )

rows = []
for key, package in sorted(library.items()):
    records = package.options.get("stress_period_data")
    rows.append({
        "key": key,
        "builder": getattr(package.builder, "__name__", type(package.builder).__name__),
        "records": len(records[0]) if isinstance(records, dict) and 0 in records else None,
        "source": pathlib.Path(package.metadata.get("source_path", "")).name or "—",
    })
pd.DataFrame(rows)

,key,builder,records,source
0,chd/base,partial,118.0,boundaries.gpkg
1,crisp/base,partial,521.0,crisp_creek.gpkg
2,dis/base,build_disv,NaN,—
3,drn/base,partial,935.0,boundaries.gpkg
4,ghb/base,partial,263.0,boundaries.gpkg
5,ic/steady,build_ic,NaN,—
6,lak/sw,_build_lak_with_tables,NaN,—
7,npf/base,ModflowGwfnpf,NaN,—
8,oc/all,build_oc,NaN,—
9,rch/background,partial,NaN,—


Twelve packages, five of them resolved from GeoPackages. `builder=partial` is the
`functools.partial(_build_named, ModflowGwfchd)` shape the list boundary conditions use.

That is everything needed to rebuild the project: the packages, and the simulation that
names them.

In [5]:
from myflopy.workspace import Project

imported = Project(WORK / "imported", name="imported")
for key, package in library.items():
    imported.add_package(key, package)
imported.add_simulation(run.spec)

print(f"{len(imported.packages)} packages, {len(imported.simulations)} simulation(s)")
imported.packages.summary()

12 packages, 1 simulation(s)


,key,package,builder,enabled,options,inputs,requires
0,chd/base,chd,flopy.ModflowGwfchd,True,"boundnames, filename, pname, save_flows, stres...",—,—
1,crisp/base,crisp,flopy.ModflowGwfriv,True,"auxiliary, boundnames, filename, pname, save_f...",—,—
2,dis/base,disv,build_disv,True,"botm, cell2d, idomain, length_units, ncpl, nla...",—,—
3,drn/base,drn,flopy.ModflowGwfdrn,True,"boundnames, filename, pname, save_flows, stres...",—,—
4,ghb/base,ghb,flopy.ModflowGwfghb,True,"auxiliary, boundnames, filename, pname, save_f...",—,—
5,ic/steady,ic,build_ic,True,strt,—,—
6,lak/sw,lak,_build_lak_with_tables,True,"boundnames, budget_filerecord, budgetcsv_filer...",—,—
7,npf/base,npf,flopy.ModflowGwfnpf,True,"icelltype, k, k33, save_flows, save_specific_d...",—,—
8,oc/all,oc,build_oc,True,"budget_filerecord, head_filerecord, saverecord",—,—
9,rch/background,rch,flopy.ModflowGwfrcha,True,"filename, fixed_cell, irch, pname, print_input...",—,—


And the check that the import is actually complete — prepare a **new** run from it.

In [6]:
t = time.perf_counter()
reimported = imported.prepare_run("reimported", "sw_mounding_flow")
reimported.build()
print(f"prepared and built in {time.perf_counter() - t:.1f} s")

model = reimported.built.built_model("tentrails").model
print("packages written:", model.get_package_list())
for name in ("chd", "ghb", "drn", "riv", "crisp"):
    print(f"   {name:6} {len(model.get_package(name).stress_period_data.get_data(0)):5d} records")

prepared and built in 1.5 s
packages written: ['DISV', 'NPF', 'STO', 'IC', 'CHD', 'GHB', 'DRN', 'RIV', 'CRISP', 'OC', 'LAK', 'RCH']
   chd      118 records
   ghb      263 records
   drn      935 records
   riv      500 records
   crisp    521 records


A run folder round-trips to a project and back to a run, with every record count intact.

What you do **not** get back is the declaration: `chd/base` comes back holding 118
resolved records, not `mf.chd.gpkg(bcs, layer='chd', head=522, ...)`. The library stores
what a package *is*, not the call that produced it. That is exactly the gap the lazy form
closes going forward — a lazily declared package stores its recipe, so the thing you get
back can move to another mesh.

---
# Part 2 — What the model actually declares

My first instinct was to guess the declarations from the GeoPackage column names. Wrong on
all three I tried, so it is worth reading the recorded provenance instead.

In [7]:
for key, package in sorted(library.items()):
    meta = package.metadata
    if meta.get("source_type") != "geopackage":
        continue
    print(f"{key:12} layer={meta['source_layer']!r}")
    for field, value in meta["fields"].items():
        print(f"     {field:12} = {value}")

chd/base     layer='chd'
     head         = 522
crisp/base   layer='crisp creek'
     conductance  = 50
     rbot         = {'minimum': None, 'offset': -1.5, 'reference': 'cell_top', 'type': 'CellSurfaceOffset'}
     stage        = {'minimum': None, 'offset': -0.1, 'reference': 'cell_top', 'type': 'CellSurfaceOffset'}
drn/base     layer='drn'
     conductance  = conductance
     elevation    = {'minimum': None, 'offset': 10, 'reference': 'cell_bottom', 'type': 'CellSurfaceOffset'}
ghb/base     layer='ghb'
     conductance  = conductance
     head         = head
riv/rock     layer='rock creek'
     conductance  = 100
     rbot         = {'minimum': None, 'offset': -1.5, 'reference': 'cell_top', 'type': 'CellSurfaceOffset'}
     stage        = {'minimum': None, 'offset': -0.1, 'reference': 'cell_top', 'type': 'CellSurfaceOffset'}


`chd`'s head is `522` — a **constant**, not a column. `drn`'s elevation is a
`CellSurfaceOffset("cell_bottom", offset=10)`. The creeks each carry **two** of them, one
for the stage and one for the river bottom.

Those are the interesting cases: an offset needs per-cell layer elevations, which is
exactly what a lazily resolved package does not have until the discretization is built.

In [8]:
shutil.copy(MODEL_ROOT / "from_usg" / "boundaries.gpkg", INPUTS / "boundaries.gpkg")
BCS = INPUTS / "boundaries.gpkg"
gpd.read_file(BCS, layer="drn")[["name", "layer", "elevation", "conductance"]]

,name,layer,elevation,conductance
0,lyr1_south,1,NaN,0.1
1,nw,1,NaN,500.0
2,lyr3_south,3,NaN,400.0
3,lyr4_south,4,NaN,500.0
4,lyr5_south,5,NaN,0.1


`elevation` is NaN on every feature — correct, because the model supplies elevation
through the offset and never reads the column. It is a decoy left over from the USG
export.

Worth knowing: if anyone ever writes `elevation="elevation"` against this file, ledger
247's guard stops them with *"feature 'lyr1_south' has nan for 'elevation'. MF6 would be
written a non-finite value and report success."* — which beats a NaN water budget under
"Normal termination".

---
# Part 3 — The grid, and the surfaces an offset resolves against

The lazy form resolves against whatever grid the build supplies, so this needs the real
one. `VoronoiGridPlus` takes vertices and cell definitions directly, so it can be rebuilt
from the stored discretization rather than by re-running Triangle.

In [9]:
disv_spec = library["dis/base"]
o = disv_spec.options

t = time.perf_counter()
vor = mf.VoronoiGridPlus(
    crs="EPSG:2927",
    verts=np.array([[float(v[1]), float(v[2])] for v in o["vertices"]]),
    iverts=[[int(i) for i in row[4:]] for row in o["cell2d"]],
    xcyc=np.array([[float(row[1]), float(row[2])] for row in o["cell2d"]]),
    name="tentrails",
)
print(f"{vor.ncpl:,} cells x {o['nlay']} layers, rebuilt in {time.perf_counter() - t:.1f} s")
print("extent:", [round(b) for b in vor.gdf_vorPolys.total_bounds])

20,323 cells x 5 layers, rebuilt in 0.8 s
extent: [1249383, 714298, 1267350, 729925]


### The layer geometry

A `CellSurfaceOffset` names a surface — `model_top`, `cell_top` or `cell_bottom` — and an
offset from it. `cell_top` and `cell_bottom` are **per layer**, so what an offset resolves
to depends on which layer the feature is in.

In [10]:
top = np.asarray(o["top"]); botm = np.asarray(o["botm"]); idomain = np.asarray(o["idomain"])

pd.DataFrame({
    "surface": ["model top"] + [f"layer {i + 1} bottom" for i in range(botm.shape[0])],
    "min": [top.min()] + list(botm.min(axis=1)),
    "mean": [top.mean()] + list(botm.mean(axis=1)),
    "max": [top.max()] + list(botm.max(axis=1)),
    "active cells": [""] + [int(idomain[i].sum()) for i in range(botm.shape[0])],
}).round(1)

,surface,min,mean,max,active cells
0,model top,264.1,528.3,711.6,
1,layer 1 bottom,261.1,517.0,700.0,20323
2,layer 2 bottom,260.1,500.2,673.2,20323
3,layer 3 bottom,258.6,467.2,665.2,20323
4,layer 4 bottom,257.1,430.6,660.0,20323
5,layer 5 bottom,60.6,133.9,525.0,20323


In [11]:
# What `CellSurfaceOffset("cell_bottom", 10)` means for one cell, per layer.
cell = 288
print(f"cell {cell}:  model top {top[cell]:7.2f}")
for layer in range(botm.shape[0]):
    cell_top = top[cell] if layer == 0 else botm[layer - 1, cell]
    print(f"   layer {layer + 1}: top {cell_top:7.2f}  bottom {botm[layer, cell]:7.2f}"
          f"   -> cell_bottom + 10 = {botm[layer, cell] + 10:7.2f}")

cell 288:  model top  395.54
   layer 1: top  395.54  bottom  392.54   -> cell_bottom + 10 =  402.54
   layer 2: top  392.54  bottom  391.54   -> cell_bottom + 10 =  401.54
   layer 3: top  391.54  bottom  385.88   -> cell_bottom + 10 =  395.88
   layer 4: top  385.88  bottom  376.23   -> cell_bottom + 10 =  386.23
   layer 5: top  376.23  bottom   76.23   -> cell_bottom + 10 =   86.23


The eager path reads these from `ModelContext.surfaces` or `grid.gdf_topbtm`. A lazily
resolved package has neither — `with_grid` drops both — so it falls back to the
discretization already built on the model. That fallback is why the offsets below resolve
at all, and it is measured against the eager answer rather than assumed.

---
# Part 4 — One boundary condition, both ways

`drn`, because it carries the offset and 935 records. The declaration is copied verbatim
from the model's own notebook. The **only** difference between the two calls is `context=`.

In [12]:
eager_context = ModelContext(grid=vor, domain=idomain, surfaces={"top": top, "botm": botm})

drn_kwargs = dict(
    layer="drn", nper=1,
    elevation=mf.CellSurfaceOffset("cell_bottom", 10),
    conductance="conductance",
    layer_field="layer", name="drn",
)

eager = mf.drn.gpkg(BCS, context=eager_context, **drn_kwargs)   # resolves NOW
lazy = mf.drn.gpkg(BCS, **drn_kwargs)                           # resolves at BUILD

print("eager:", len(eager.options["stress_period_data"][0]), "records, pinned to",
      eager.metadata.get("grid_pin"))
print("lazy :", sorted(lazy.inputs), "->", package_move(lazy))
print("       recipe:", lazy.inputs["stress_period_data"])

eager: 935 records, pinned to 9bff04dd4b9fedba
lazy : ['stress_period_data'] -> ('rebuilt', 'carries a recipe, so it resolves against the new mesh')
       recipe: BoundarySource(path=PosixPath('/tmp/lazy_gpkg_4enxynub/inputs/boundaries.gpkg'), external=False, metadata={}, package='drn', nper=1, values={'elevation': CellSurfaceOffset(reference='cell_bottom', offset=10, minimum=None), 'conductance': 'conductance'}, layer='drn', name_field='name', layer_field='layer', period_field=None, layer_base=1, period_base=0, edges_only=False, boundnames=True)


The eager one holds records and a grid pin. The lazy one holds a `BoundarySource` recipe
(ledger 277) and reports `rebuilt`, meaning it can move to another mesh under its own
power where the eager one would read `stale`.

Now build it. Note the context here carries **no domain and no surfaces** — the normal
situation after `with_grid`.

In [13]:
disv = mf.disv(
    nlay=o["nlay"], ncpl=o["ncpl"], nvert=o["nvert"],
    vertices=o["vertices"], cell2d=o["cell2d"],
    top=o["top"], botm=o["botm"], idomain=o["idomain"],
)

simulation = mf.simulation(
    mf.gwf("m", context=ModelContext(grid=vor), packages=[disv, lazy]),   # bare context
    name="s", packages=[mf.tdis(nper=1)],
)
t = time.perf_counter()
built = simulation.build(WORK / "ws_drn")
print(f"built in {time.perf_counter() - t:.2f} s")

resolved = built.simulation.get_model("m").get_package("drn").stress_period_data.get_data(0)
declared = eager.options["stress_period_data"][0]
print(f"records: lazy {len(resolved)}  eager {len(declared)}")
print("same cells     :", [tuple(r)[0] for r in resolved] == [tuple(r)[0] for r in declared])
print("same elevations:", np.allclose([tuple(r)[1] for r in resolved],
                                      [tuple(r)[1] for r in declared]))
print("same boundnames:", [str(tuple(r)[-1]) for r in resolved]
                       == [str(tuple(r)[-1]) for r in declared])

built in 1.48 s
records: lazy 935  eager 935
same cells     : True
same elevations: True
same boundnames: True


In [14]:
# What the offset resolved to, against the layer geometry above.
pd.DataFrame({
    "cellid": [tuple(r)[0] for r in resolved[:6]],
    "elevation": [round(tuple(r)[1], 2) for r in resolved[:6]],
    "cell bottom": [round(botm[tuple(r)[0][0], tuple(r)[0][1]], 2) for r in resolved[:6]],
    "conductance": [tuple(r)[2] for r in resolved[:6]],
    "boundname": [str(tuple(r)[-1]) for r in resolved[:6]],
})

,cellid,elevation,cell bottom,conductance,boundname
0,"(0, 288)",402.54,392.54,0.1,lyr1_south
1,"(0, 463)",410.24,400.24,0.1,lyr1_south
2,"(0, 464)",399.28,389.28,0.1,lyr1_south
3,"(0, 538)",486.59,476.59,0.1,lyr1_south
4,"(0, 539)",486.15,476.15,0.1,lyr1_south
5,"(0, 580)",369.49,359.49,0.1,lyr1_south


`elevation` is exactly `cell bottom + 10` in every row — the offset resolved against the
discretization the build supplied, not against anything the declaration carried.

### An aside that turned out to matter

The *eager* call needed `surfaces=` spelled out. Without it, `CellSurfaceOffset` raises
*"ModelContext.surfaces or grid.gdf_topbtm is required"*. The *lazy* call did not. The new
path is, here, the more forgiving one.

---
# Part 5 — All seven helpers

Five of the seven list boundary conditions appear in this model. The counts to beat are
the ones already stored in the run's library — from a model that was prepared, written and
executed long before any of this existed.

In [15]:
shutil.copy(DATA_ROOT / "crisp_creek.gpkg", INPUTS / "crisp_creek.gpkg")
shutil.copy(DATA_ROOT / "rock creek.gpkg", INPUTS / "rock_creek.gpkg")

creek = dict(nper=1, name_field="name", layer_field="layer",
             stage=mf.CellSurfaceOffset("cell_top", -0.1),
             rbot=mf.CellSurfaceOffset("cell_top", -1.5))

cases = [
    ("chd",   mf.chd, BCS, dict(layer="chd", nper=1, head=522, layer_field="layer", name="chd")),
    ("ghb",   mf.ghb, BCS, dict(layer="ghb", nper=1, head="head", conductance="conductance",
                                layer_field="layer", name="ghb")),
    ("drn",   mf.drn, BCS, drn_kwargs),
    ("riv",   mf.riv, INPUTS / "rock_creek.gpkg",
              dict(layer="rock creek", name="riv", conductance=100, **creek)),
    ("crisp", mf.riv, INPUTS / "crisp_creek.gpkg",
              dict(layer="crisp creek", name="crisp", conductance=50, **creek)),
]

def compare(name, helper, source, kwargs, stored=None):
    eager_spec = helper.gpkg(source, context=eager_context, **kwargs)
    lazy_spec = helper.gpkg(source, **kwargs)
    sim = mf.simulation(
        mf.gwf("m", context=ModelContext(grid=vor), packages=[disv, lazy_spec]),
        name="s", packages=[mf.tdis(nper=1)],
    )
    t = time.perf_counter()
    model = sim.build(WORK / f"ws_{name}").simulation.get_model("m")
    elapsed = time.perf_counter() - t

    got = model.get_package(name).stress_period_data.get_data(0)
    want = eager_spec.options["stress_period_data"][0]
    # A record is (cellid, value..., boundname): compare numbers with a tolerance and the
    # name as a string. `np.allclose` on the whole row raises DTypePromotionError, which
    # is the boundname pointing out that it is not a number.
    numeric = [i for i, v in enumerate(tuple(want[0])) if isinstance(v, (int, float))]
    return {
        "package": name, "move": package_move(lazy_spec)[0],
        "eager": len(want), "lazy": len(got), "stored": stored,
        "cells": [tuple(r)[0] for r in got] == [tuple(r)[0] for r in want],
        "values": all(np.allclose([tuple(r)[i] for r in got], [tuple(r)[i] for r in want])
                      for i in numeric),
        "names": [str(tuple(r)[-1]) for r in got] == [str(tuple(r)[-1]) for r in want],
        "build s": round(elapsed, 2),
    }

stored_counts = {k.split("/")[0]: r["records"] for k, r in
                 ((k, {"records": len(p.options["stress_period_data"][0])})
                  for k, p in library.items()
                  if isinstance(p.options.get("stress_period_data"), dict))}
pd.DataFrame([compare(n, h, s, k, stored_counts.get(n)) for n, h, s, k in cases])

,package,move,eager,lazy,stored,cells,values,names,build s
0,chd,rebuilt,118,118,118,True,True,True,1.36
1,ghb,rebuilt,263,263,263,True,True,True,1.38
2,drn,rebuilt,935,935,935,True,True,True,1.46
3,riv,rebuilt,500,500,500,True,True,True,1.37
4,crisp,rebuilt,521,521,521,True,True,True,1.31


Five for five, every count matching what the model stored.

### The two this model does not use

`wel` and `evt` have no GeoPackage source here, so there is nothing real to check them
against. Rather than skip them, here they are on **synthetic features over the real
grid** — three wells and an ET polygon. Clearly made up; the grid is not.

In [16]:
bounds = vor.gdf_vorPolys.total_bounds
def at(fx, fy):
    from shapely.geometry import Point
    return Point(bounds[0] + (bounds[2] - bounds[0]) * fx,
                 bounds[1] + (bounds[3] - bounds[1]) * fy)

SYNTH = INPUTS / "synthetic.gpkg"
gpd.GeoDataFrame({"name": ["w1", "w2", "w3"], "layer": [1, 2, 3],
                  "rate": [-500.0, -250.0, -100.0]},
                 geometry=[at(0.3, 0.5), at(0.5, 0.5), at(0.7, 0.5)],
                 crs=vor.crs).to_file(SYNTH, layer="wells", driver="GPKG")

from shapely.geometry import box as _box
gpd.GeoDataFrame({"name": ["field"], "layer": [1], "rate": [0.002], "depth": [8.0]},
                 geometry=[_box(bounds[0] + (bounds[2] - bounds[0]) * 0.4,
                                bounds[1] + (bounds[3] - bounds[1]) * 0.4,
                                bounds[0] + (bounds[2] - bounds[0]) * 0.6,
                                bounds[1] + (bounds[3] - bounds[1]) * 0.6)],
                 crs=vor.crs).to_file(SYNTH, layer="et", driver="GPKG")

pd.DataFrame([
    compare("wel", mf.wel, SYNTH,
            dict(layer="wells", nper=1, rate="rate", layer_field="layer", name="wel")),
    compare("evt", mf.evt, SYNTH,
            dict(layer="et", nper=1, surface=mf.CellSurfaceOffset("cell_top", 0.0),
                 rate="rate", depth="depth", layer_field="layer", name="evt")),
])

,package,move,eager,lazy,stored,cells,values,names,build s
0,wel,rebuilt,3,3,None,True,True,True,1.35
1,evt,rebuilt,272,272,None,True,True,True,1.34


All seven list helpers agree with their eager equivalents. Three wells resolve to three
cells (a point has no extent to split); the ET polygon covers 272.

### What is *not* lazy

Worth being explicit, because this model uses two of them:

| package | in this model | lazy? |
|---|---|---|
| `chd` `ghb` `drn` `riv` `wel` `rch.gpkg` `evt.gpkg` | 5 of 7 | **yes** |
| `rch.array` / `evt.array` | `rch/background` | not yet |
| `lak` | `lak/sw` | not yet |
| `uzf` `sfr` `hfb` | — | not yet |

Everything in the lower rows still resolves at declaration and still reads `stale` on a
grid move. They are the remaining B2 work.

In [17]:
# The model's own packages, as `move_report` sees them.
from myflopy.specs import GRID_PIN

pd.DataFrame([
    {"key": key,
     "verdict": package_move(package)[0],
     "pinned": package.metadata.get(GRID_PIN) is not None,
     "why": package_move(package)[1][:52]}
    for key, package in sorted(library.items())
])

,key,verdict,pinned,why
0,chd/base,stale,True,its records are pinned to mesh 9bff04dd4b9fedba
1,crisp/base,stale,True,its records are pinned to mesh 9bff04dd4b9fedba
2,dis/base,stale,False,it describes the old mesh itself (hand-written...
3,drn/base,stale,True,its records are pinned to mesh 9bff04dd4b9fedba
4,ghb/base,stale,True,its records are pinned to mesh 9bff04dd4b9fedba
5,ic/steady,unchanged,False,names no cells
6,lak/sw,unchanged,False,names no cells
7,npf/base,unchanged,False,names no cells
8,oc/all,unchanged,False,names no cells
9,rch/background,unchanged,False,names no cells


### A finding worth acting on

`lak/sw` and `rch/background` read **`unchanged`** — the verdict that means *moves
untouched* — and carry no pin. That is not because they are mesh-independent. They are
both thoroughly grid-bound: LAK's cell numbers live in `connectiondata`, and this model's
recharge resolves `irch="top_active"` to a per-cell array.

They are unpinned because **they were declared before the pin existed**. Ledger 267
(2026-09-15) extended pinning to `lak`, `hfb` and the array forms of `rch`/`evt`; this run
was prepared on 2026-09-14. A pin is stamped at declaration, so it cannot be applied
retroactively to a package already on disk.

The consequence is concrete: if you took the project imported in Part 1 and moved it to a
new grid, `chd`/`ghb`/`drn`/`riv`/`crisp` would be refused as `stale` — correctly — while
`lak` and `rch` would be carried across in silence, with cell numbers belonging to the old
mesh. That is exactly the failure ledger 267 was written to stop.

**The fix is to re-declare them**, which re-stamps the pin:

In [18]:
# Re-declaring an unpinned package is what stamps it. Shown on `rch` with the model's own
# recharge array -- the declaration is from the source notebook, the array from the run.
recharge_now = library["rch/background"].options["recharge"]
restamped = mf.rch.array(context=eager_context, recharge=recharge_now, name="rch")

print("stored    :", package_move(library["rch/background"]))
print("re-declared:", package_move(restamped))
print("pin        :", restamped.metadata.get(GRID_PIN))

stored    : ('unchanged', 'names no cells')
re-declared: ('stale', 'its records are pinned to mesh 9bff04dd4b9fedba')
pin        : 9bff04dd4b9fedba


---
# Part 6 — Does a run still rebuild without its GeoPackage?

This is where the lazy form could plausibly have made things *worse*. A resolved package
holds records; a lazy one holds a path. If a run froze the recipe, reopening it a year
later would send it back to a file that may have moved — the dependence the frozen
snapshot exists to remove (ledger 251).

So: prepare a run, delete the source, rebuild from the snapshot alone.

In [19]:
from myflopy.project.freeze import thaw_simulation

private = WORK / "private"; private.mkdir()
shutil.copy(BCS, private / "boundaries.gpkg")      # a copy this cell may delete

project = Project(WORK / "proj", name="tentrails")
project.add_simulation(mf.simulation(
    mf.gwf("m", context=ModelContext(grid=vor),
           packages=[disv, mf.drn.gpkg(private / "boundaries.gpkg", **drn_kwargs)]),
    name="s", packages=[mf.tdis(nper=1)],
))

frozen_run = project.prepare_run("r", "s")
frozen_run.build()
before = frozen_run.built.built_model("m").model.get_package("drn").stress_period_data.get_data(0)

snapshot = frozen_run.workspace / ".myflopy" / "frozen.npz"
print(f"frozen.npz: {snapshot.stat().st_size / 1e6:.2f} MB, {len(before)} drn records")
pd.DataFrame(frozen_run.resolution)

frozen.npz: 3.98 MB, 935 drn records


,model,package,how,why,resolved_records
0,m,disv,stale,it describes the old mesh itself (hand-written...,NaN
1,m,drn,rebuilt,"carries a recipe, so it resolves against the n...",935.0


In [20]:
shutil.rmtree(private)                              # the GeoPackage is now gone

thawed = thaw_simulation(frozen_run.workspace)
rebuilt = thawed.build_flopy(workspace=WORK / "again")
after = rebuilt.built_model("m").model.get_package("drn").stress_period_data.get_data(0)

print(f"rebuilt from the snapshot alone: {len(after)} records")
print("same cells :", [tuple(r)[0] for r in after] == [tuple(r)[0] for r in before])
print("same values:", np.allclose([tuple(r)[1] for r in after], [tuple(r)[1] for r in before]))

rebuilt from the snapshot alone: 935 records
same cells : True
same values: True


Two details in the resolution table that are easy to miss:

- `drn` reads `rebuilt` with `resolved_records: 935`, because a boundary condition
  resolves to *records*; a discretization would report `resolved_ncpl` instead;
- the **frozen** `drn` spec is pinned and would read `stale` — correct, because something
  already resolved is no longer free to move. The **declared** spec keeps its recipe and
  stays `rebuilt`. Two objects, each staying itself.

---
# Part 7 — What does it cost?

Laziness moves resolution from declaration to build, and a model is built more often than
it is declared. Worth measuring rather than assuming.

In [21]:
timings = {}
for label, packages in (
    ("disv alone", [disv]),
    ("disv + lazy drn (935 records)", [disv, mf.drn.gpkg(BCS, **drn_kwargs)]),
    ("disv + all five", [disv] + [h.gpkg(s, **k) for _, h, s, k in cases]),
):
    sim = mf.simulation(mf.gwf("m", context=ModelContext(grid=vor), packages=packages),
                        name="s", packages=[mf.tdis(nper=1)])
    t = time.perf_counter()
    sim.build(WORK / "ws_cost")
    timings[label] = time.perf_counter() - t

for label, seconds in timings.items():
    print(f"  {label:32} {seconds:5.2f} s")
print(f"\n  all five boundary conditions cost "
      f"{timings['disv + all five'] - timings['disv alone']:.2f} s "
      f"on a {timings['disv alone']:.2f} s build")
print(f"  declaring one is essentially free: ", end="")
t = time.perf_counter(); mf.drn.gpkg(BCS, **drn_kwargs)
print(f"{(time.perf_counter() - t) * 1000:.1f} ms")

  disv alone                        1.35 s
  disv + lazy drn (935 records)     1.37 s
  disv + all five                   1.68 s

  all five boundary conditions cost 0.33 s on a 1.35 s build
  declaring one is essentially free: 0.2 ms


The build is dominated by handing 20,323 cells and 41,813 vertices to FloPy. Resolving all
five boundary conditions on top of that is a fraction of it, and declaring one is free —
it only records the recipe.

---
## What this notebook does *not* show

A green notebook invites more confidence than it earns, so:

- **One mesh.** Every check uses the model's own 20,323-cell grid. The *point* of the lazy
  form is that one declaration works on a **different** mesh, and that is only tested
  against the synthetic pair in `tests/test_lazy_gpkg.py`. Remapping this model needs a
  second grid to remap to, which is the next piece of work.
- **No MF6 run.** Nothing here executes MODFLOW. It shows the written package matches what
  the eager path writes — the right comparison, since the eager path produced the runs you
  already trust.
- **List boundary conditions only.** `rch`/`evt` in array form, `lak`, `uzf`, `sfr` and
  `hfb` all still resolve eagerly (Part 5).
- **`wel` and `evt` on invented features.** Real grid, made-up boundaries. They show the
  helpers agree with themselves, not that they are right about anything in this model.

In [22]:
shutil.rmtree(WORK, ignore_errors=True)
print("cleaned up:", WORK)
print("the model directory was only ever read from.")

cleaned up: /tmp/lazy_gpkg_4enxynub
the model directory was only ever read from.
